# 0914 11일차

## 0. 파이썬 문법 - 파일명 문자열 만들기

체크포인트 파일명에 날짜와 epoch·val_loss를 넣으려고 문자열을 조합함

In [ ]:
filepath = 'k30_0914_1323_{epoch:04d}-{val_loss:.4f}.keras'

print(filepath)                                          # k30_0914_1323_{epoch:04d}-{val_loss:.4f}.keras  <- 아직 빈칸
print(filepath.format(epoch=9, val_loss=0.543210987))    # k30_0914_1323_0009-0.5432.keras

print('{val_loss:4f}'.format(val_loss=0.543210987))      # 0.543211  <- . 이 빠지면 소수점 6자리

### 0-1. datetime.datetime - 모듈 안의 클래스

모듈 `datetime` 안에 같은 이름의 클래스 `datetime`이 있어서 `datetime`이 두 번 붙음

**가져오는 방법**
1. `import datetime` → `datetime.datetime.now()` : 모듈을 가져와 그 안의 클래스를 씀
2. `from datetime import datetime` → `datetime.now()` : 클래스를 바로 가져옴 (5일차 §0-3)

**규칙**
- `now()`가 돌려주는 것은 `datetime` 객체임
- `strftime()`을 거쳐야 문자열이 되고, 그래야 `join`으로 이어붙일 수 있음

### 0-2. ''.join() - 문자열 이어붙이기

리스트의 문자열들을 구분자를 끼워 하나로 합치는 메서드

```python
''.join(['C:/study/_save/keras30/', 'k30_', '0914_1323_', '{epoch:04d}-{val_loss:.4f}.keras'])
# 'C:/study/_save/keras30/k30_0914_1323_{epoch:04d}-{val_loss:.4f}.keras'
```

**규칙**
1. 앞의 `''`가 원소 사이에 끼울 구분자임 → `'_'.join(['a', 'b'])`면 `'a_b'`
2. `path + 'k30_' + date + filename`과 결과가 같음
3. 원소가 모두 문자열이어야 함 → `strftime` 전의 `datetime` 객체를 넣으면 `TypeError`

### 0-3. 포맷 지정자

`{이름:형식}` 자리에 값을 정해진 형식으로 채워 넣는 표기

| 표기 | 뜻 | `9` / `0.543210987` 을 넣으면 |
|---|---|---|
| `{epoch:04d}` | 정수, 4자리, 빈 자리는 0 | `0009` |
| `{val_loss:.4f}` | 실수, 소수점 아래 4자리 | `0.5432` |
| `{val_loss:4f}` | 실수, 전체 폭 4, 소수점 아래는 기본 6자리 | `0.543211` |

**규칙**
1. 소수점 아래 4자리는 `.4f` (`4f`는 전체 폭이라 다른 뜻)
2. epoch를 `04d`로 채우면 파일명 이름순이 epoch 순서와 같아짐 (`0009` < `0052`)

## 1. 가중치 저장과 불러오기 (save_weights / load_weights)

모델 구조 없이 가중치만 파일로 저장하는 기능 (`keras29_5_save_weights` / `keras29_6_load_weights`)

```python
model.save_weights(path + 'keras29_5_save_weights1.weights.h5')   # 모델 구성 직후 (fit 전)
model.compile(loss="mse", optimizer="adam")
model.fit(...)
model.save_weights(path + 'keras29_5_save_weights2.weights.h5')   # fit 후
```

**규칙**
- 확장자는 `.weights.h5`로 끝나야 함. 아니면 `ValueError: The filename must end in .weights.h5`

**model.save()와의 차이**

| | `model.save()` (10일차 §5) | `model.save_weights()` |
|---|---|---|
| 확장자 | `.keras` | `.weights.h5` |
| 모델 구조 | 들어 있음 | 없음 |
| 가중치 | 들어 있음 | 들어 있음 |
| compile 정보 | 들어 있음 | 없음 |
| 불러오기 | `model = load_model(...)` | 모델 구성 → `model.load_weights(...)` → `compile` |

### 1-1. load_weights 순서

가중치 파일에는 구조가 없어서 같은 구조의 모델을 먼저 만들고 그 위에 불러와야 함

**순서**
1. 저장할 때와 같은 구조로 모델을 구성함
2. `load_weights`로 가중치를 불러옴
3. compile 정보도 없으므로 다시 `compile`함
4. 평가함

```python
model = Sequential()
model.add(Dense(5, input_dim=8))
...
model.add(Dense(1))

model.load_weights(path + 'keras29_5_save_weights2.weights.h5')
model.compile(loss="mse", optimizer="adam")     # compile 정보도 없으니 다시
loss = model.evaluate(x_test, y_test)
```

- 모델 구성 없이 가중치만 불러오면 에러가 남
- `29_6`이 모델 구성 코드를 그대로 두고 `fit`만 주석 처리한 이유임

### 1-2. 저장 시점에 따른 가중치

| 파일 | 저장 시점 | 들어 있는 가중치 |
|---|---|---|
| `weights1` | 모델 구성 직후 | 무작위 초기값 |
| `weights2` | `fit` 후 | 학습된 값 |

- 모델 구성 단계에서 저장한 가중치를 불러오는 것은 의미가 없음 → 그래서 `29_6`은 `weights2`를 불러옴

## 2. ModelCheckpoint

훈련하면서 가장 좋았던 지점의 모델을 파일로 저장하는 콜백. 그 저장 지점을 체크포인트라고 함

(`keras30_ModelCheckPoint1` / `keras30_ModelCheckPoint2_load`)

**필요한 이유**
- 지금까지는 훈련이 끝난 뒤의 가중치만 저장할 수 있었음
- 훈련 중간에 가장 좋았던 지점을 파일로 남기려면 체크포인트가 필요함

```python
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

mcp = ModelCheckpoint(
    monitor='val_loss',
    mode='auto',
    save_best_only=True,                    # 개선됐을 때만 저장
    filepath=path + 'keras30_mcp1.keras',
    verbose=1,
)

hist = model.fit(x_train, y_train, epochs=1000, batch_size=32,
                 validation_split=0.2,
                 callbacks=[es, mcp],        # EarlyStopping과 같이 넘김
                 )
```

**주요 파라미터**
1. `monitor` : 기준 지표. EarlyStopping과 같이 `val_loss`
2. `mode` : `min` / `max` / `auto`
3. `save_best_only` : `True`면 `monitor`가 개선된 epoch에만 저장
4. `filepath` : 저장 경로. `.keras`면 모델 전체가 저장됨 (파일명 만들기는 §0)
5. `verbose` : `1`이면 저장할 때마다 로그 출력

### 2-1. 실행 로그 읽기

```
Epoch 21: val_loss improved from 0.54551 to 0.52847, saving model to C:\study\_save\keras30/keras30_mcp1.keras
Epoch 21: finished saving model to C:\study\_save\keras30/keras30_mcp1.keras
Epoch 22: val_loss did not improve from 0.52847
...
Epoch 51: early stopping
Restoring model weights from the end of the best epoch: 21.
```

- 개선되면 `improved from A to B, saving model`, 아니면 `did not improve`
- 21 + `patience=30` = 51에서 멈추고, EarlyStopping이 21번째 가중치로 되돌림
- 마지막으로 저장된 체크포인트도 21번째 epoch임

### 2-2. restore_best_weights와 체크포인트의 차이

| | `EarlyStopping(restore_best_weights=True)` | `ModelCheckpoint(save_best_only=True)` |
|---|---|---|
| 저장 위치 | 메모리 (실행 중인 `model`) | 파일 |
| 언제 | 훈련이 끝날 때 한 번 되돌림 | 개선될 때마다 저장 |
| 프로그램 종료 후 | 사라짐 | 남음 |

- 둘 다 한 번의 `fit` 안에서 `val_loss`가 가장 낮았던 epoch가 기준임
- `restore_best_weights`는 조기 종료가 되지 않고 `epochs`를 끝까지 돌아도, 훈련 끝에 최고 지점으로 되돌림

### 2-3. 체크포인트 불러오기

`keras30_ModelCheckPoint2_load`

```python
model = load_model(path + 'keras30_mcp1.keras')    # 모델 구성 · compile · 가중치 전부
loss = model.evaluate(x_test, y_test)              # compile 없이 바로 평가
```

- 체크포인트 `.keras`에는 모델 구성, compile 정보, 학습된 가중치가 모두 들어 있어 불러오기만 하면 됨
- `30_2`는 모델 구성, `compile`, `fit` 코드를 모두 주석 처리하고 데이터 처리와 평가만 남김

## 3. Dropout

훈련 중에 층의 노드 일부를 랜덤하게 꺼서(출력을 0으로 만들어) 과적합을 줄이는 층

```python
from tensorflow.keras.layers import Dropout

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))        # 이 층 출력의 20%를 랜덤하게 0으로
```

**Dropout이 과적합을 줄이는 이유**
- 매번 다른 노드가 꺼지므로, 모델이 특정 노드에 의존하지 않게 됨

**규칙**
1. 꺼지는 노드는 배치(가중치 한 번 갱신)마다 랜덤하게 바뀜 → 한 epoch 안에서도 계속 바뀜
2. `evaluate`, `predict`에서는 끄지 않고 전체 노드를 씀
3. Dropout 층은 파라미터가 없음 → 노드를 끄기만 하고 자체적으로 계산하는 가중치가 없기 때문

## 4. 함수형 모델 (Input / Model)

층을 변수로 받아 앞 층과 직접 연결해서 만드는 모델 (`keras34_function00`)

```python
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input

# 순차형 - add로 쌓음
model = Sequential()
model.add(Dense(10, input_shape=(3,)))
model.add(Dropout(0.2))
model.add(Dense(9))
model.add(Dropout(0.2))
model.add(Dense(1))

# 함수형 - 층 뒤에 (앞 층 변수)를 붙여 연결
input1 = Input(shape=(3,))
dense1 = Dense(10, name='1st')(input1)
drop1 = Dropout(0.2)(dense1)
dense2 = Dense(9)(drop1)
drop2 = Dropout(0.2)(dense2)
output1 = Dense(1)(drop2)

model2 = Model(inputs=input1, outputs=output1)
```

**Sequential과의 차이**

| | `Sequential` | 함수형 `Model` |
|---|---|---|
| 입력 | 첫 층의 `input_shape` | `Input(shape=...)` 층을 따로 만듦 |
| 층 연결 | `add` 순서대로 | `Dense(...)(앞 층)`으로 직접 연결 |
| 모델 정의 | 처음에 `Sequential()` | 마지막에 `Model(inputs=, outputs=)` |

**함수형 모델의 규칙**
1. `Dense(10)`까지가 층을 만드는 부분이고, 뒤의 `(input1)`이 앞 층과 연결하는 부분임
2. 층만 연결해서는 모델이 아님 → `Model(inputs=input1, outputs=output1)`로 시작과 끝을 지정해야 함
3. `name='1st'`은 층 이름. 쓰지 않으면 `dense_3`처럼 자동으로 붙음

- 더 유연하거나 복잡한 모델을 만들 때 함수형을 씀

### 4-1. summary 비교 - 파라미터는 같음

| 층 | `Sequential` | 함수형 | Param |
|---|---|---|---|
| 입력 | (표시 안 됨) | `input_layer_1 (InputLayer)` `(None, 3)` | 0 |
| `Dense(10)` | `dense` | `1st` | 40 |
| `Dropout(0.2)` | `dropout` | `dropout_2` | 0 |
| `Dense(9)` | `dense_1` | `dense_3` | 99 |
| `Dropout(0.2)` | `dropout_1` | `dropout_3` | 0 |
| `Dense(1)` | `dense_2` | `dense_4` | 10 |
| 합계 | 149 | 149 | |

- 함수형은 입력층이 `InputLayer`로 summary에 따로 나오고, 가중치가 없어 Param 0임
- 파라미터는 둘 다 149개 (`(3+1)×10`, `(10+1)×9`, `(9+1)×1`, 9일차 §2-1) → 기능은 같고 표현 방식만 다름
- 함수형의 층 번호가 `dropout_2`, `dense_3`부터인 것은 같은 파일의 `Sequential`이 앞 번호를 먼저 썼기 때문임